# NaN Edge Cases: Legacy vs v1

Development notebook investigating how NaN behaves across linopy operations under both conventions.

**Core principle (v1):** NaN means "absent term" — not a numeric value. It enters only through structural operations (`shift`, `where`, `reindex`, `mask=`) and propagates via IEEE semantics. Absent terms don't poison valid terms at the same coordinate.

1. [Sources of NaN](#sources-of-nan)
2. [isnull detection](#isnull-detection)
3. [Arithmetic on shifted expressions](#arithmetic-on-shifted-expressions)
4. [Combining expressions with absent terms](#combining-expressions-with-absent-terms)
5. [Constraints from expressions with NaN](#constraints-from-expressions-with-nan)
6. [Reviving absent slots with fillna and fill_value](#reviving-absent-slots)
7. [FILL_VALUE internals](#fill_value-internals)

In [ ]:
import warnings

import pandas as pd
import xarray as xr

import linopy
from linopy import Model
from linopy.config import LinopyDeprecationWarning
from linopy.expressions import FILL_VALUE

warnings.filterwarnings("ignore", category=LinopyDeprecationWarning)

MindOpt 2.2.0 | 2e28db43, Aug 29 2025, 14:27:12 | arm64 - macOS 26.2
Start license validation (current time : 14-MAR-2026 18:35:55 UTC+0100).
[WARN ] No license file is found.
[ERROR] No valid license was found. Please visit https://opt.aliyun.com/doc/latest/en/html/installation/license.html to apply for and set up your license.
License validation terminated. Time : 0.000s



In [ ]:
def make_model():
    m = Model()
    time = pd.RangeIndex(5, name="time")
    x = m.add_variables(lower=0, coords=[time], name="x")
    return m, x

---

## Sources of NaN

### shift

`.shift()` is the primary structural source of NaN. It shifts data along a dimension, creating a gap filled with `FILL_VALUE` (`vars=-1`, `coeffs=NaN`, `const=NaN`).

In [ ]:
m, x = make_model()
expr = 2 * x + 10
expr.shift(time=1)

LinearExpression [time: 5]:
---------------------------
[0]: None
[1]: +2 x[0] + 10
[2]: +2 x[1] + 10
[3]: +2 x[2] + 10
[4]: +2 x[3] + 10

In [ ]:
# Variables also support shift — labels get -1 sentinel, bounds get NaN
x.shift(time=1)

Variable (time: 5) - 1 masked entries
-------------------------------------
[0]: None
[1]: x[0] ∈ [0, inf]
[2]: x[1] ∈ [0, inf]
[3]: x[2] ∈ [0, inf]
[4]: x[3] ∈ [0, inf]

### roll

`.roll()` is circular — values wrap around, no NaN introduced.

In [ ]:
m, x = make_model()
(2 * x + 10).roll(time=1)

LinearExpression [time: 5]:
---------------------------
[0]: +2 x[4] + 10
[1]: +2 x[0] + 10
[2]: +2 x[1] + 10
[3]: +2 x[2] + 10
[4]: +2 x[3] + 10

### where

`.where(cond)` masks slots where the condition is False → `vars=-1, coeffs=NaN, const=NaN`.

In [ ]:
m, x = make_model()
mask = xr.DataArray([True, True, False, False, True], dims=["time"])
(2 * x + 10).where(mask)

LinearExpression [time: 5]:
---------------------------
[0]: +2 x[0] + 10
[1]: +2 x[1] + 10
[2]: None
[3]: None
[4]: +2 x[4] + 10

### reindex

`.reindex()` expands or shrinks coordinates. New coordinates get `FILL_VALUE`.

In [ ]:
m, x = make_model()
expr = 2 * x + 10

# Expand to a larger index — new positions [5, 6] are absent
expr.reindex({"time": pd.RangeIndex(7, name="time")})

LinearExpression [time: 7]:
---------------------------
[0]: +2 x[0] + 10
[1]: +2 x[1] + 10
[2]: +2 x[2] + 10
[3]: +2 x[3] + 10
[4]: +2 x[4] + 10
[5]: None
[6]: None

---

## isnull detection

`isnull()` checks: `(vars == -1).all(helper_dims) & const.isnull()`

Both conditions must be true — a slot is only "absent" if there are no variables AND no constant. This distinguishes "absent" from "valid expression with zero constant".

In [ ]:
m, x = make_model()
shifted = (2 * x + 10).shift(time=2)
shifted.isnull()

<xarray.DataArray (time: 5)> Size: 5B
array([ True,  True, False, False, False])
Coordinates:
  * time     (time) int64 40B 0 1 2 3 4

---

## Arithmetic on shifted expressions

When you do arithmetic on an expression with absent slots (from `shift`/`where`/`reindex`):

- **Addition/subtraction**: fills const with 0 (additive identity) before adding. This preserves associativity: `(a + b) + c == a + (b + c)`.
- **Multiplication/division**: NaN propagates. No implicit fill — the "right" neutral element depends on context (0 kills, 1 preserves).

Legacy mode fills all NaN with neutral elements for both add and mul.

In [ ]:
linopy.options["arithmetic_convention"] = "legacy"
m, x = make_model()
shifted = (2 * x + 10).shift(time=1)

# Legacy: NaN const filled with 0, then +5 = 5. Slot looks alive!
shifted + 5

LinearExpression [time: 5]:
---------------------------
[0]: +5
[1]: +2 x[0] + 15
[2]: +2 x[1] + 15
[3]: +2 x[2] + 15
[4]: +2 x[3] + 15

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()
shifted = (2 * x + 10).shift(time=1)

# v1: addition fills const with 0 (additive identity), then adds 5
shifted + 5

In [ ]:
# v1: multiplication propagates NaN — absent stays absent
shifted * 3

---

## Combining expressions with absent terms

When two expressions are merged (e.g., `x + y.shift(1)`), each term is concatenated along the `_term` dimension. The constant is summed with `skipna=True` — NaN from one operand does **not** poison the other.

**Key rule: absent terms don't poison valid terms at the same coordinate.**

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()
y = m.add_variables(lower=0, coords=[pd.RangeIndex(5, name="time")], name="y")

# x is valid everywhere, y.shift(1) is absent at time=0
# → time=0 still has x's term, only y's term is absent
x + (1 * y).shift(time=1)

In [ ]:
# Shifted constant is LOST at the gap:
# (y+5).shift makes the ENTIRE expression absent at time=0 — including its constant.
# Only the outer +5 survives. time=1..4 get const=10 (shifted 5 + outer 5).
x + (1 * y + 5).shift(time=1) + 5

LinearExpression [time: 5]:
---------------------------
[0]: +1 x[0] + 5
[1]: +1 x[1] + 1 y[0] + 10
[2]: +1 x[2] + 1 y[1] + 10
[3]: +1 x[3] + 1 y[2] + 10
[4]: +1 x[4] + 1 y[3] + 10

In [ ]:
# Both expressions shifted — time=0 is fully absent (all terms absent AND const=NaN)
result = (1 * x).shift(time=1) + (1 * y).shift(time=1)
print("isnull:", result.isnull().values)
result

### Summary

| Expression | const at time=0 | isnull at time=0 | Why |
|---|---|---|---|
| `x + y.shift(1)` | 0 | False | y's term absent, x valid, const sum skips NaN |
| `x + y.shift(1) + 5` | 5 | False | Same, then +5 on const |
| `x + (y+5).shift(1) + 5` | 5 | False | Shifted const (5) is lost — only outer +5 survives |
| `x.shift(1) + y.shift(1)` | NaN | True | All terms absent AND all consts NaN → fully absent |

### Legacy vs v1: scalar arithmetic on shifted expressions

| | Legacy | v1 |
|---|---|---|
| `shifted + 5` at absent slot | const=5 (alive) | const=5 (alive, additive identity fill) |
| `shifted * 3` at absent slot | coeffs=0, const=0 (alive) | coeffs=NaN, const=NaN (absent) |
| `shifted - 5` at absent slot | const=-5 (alive) | const=-5 (alive, additive identity fill) |
| `shifted / 2` at absent slot | coeffs=0, const=0 (alive) | coeffs=NaN, const=NaN (absent) |

**v1 rule:** addition/subtraction use 0 as additive identity to fill NaN const. Multiplication/division propagate NaN — use `.fillna(value)` or `.mul(v, fill_value=)` for explicit control.

---

## Constraints from expressions with NaN

Absent slots in expressions propagate to constraint RHS. The preferred approach is to avoid NaN entirely using `isel` + positional alignment, or to filter with `.sel()`.

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()

# Preferred: isel + override avoids NaN entirely
x_now = 1 * x.isel(time=slice(1, None))
x_prev = 1 * x.isel(time=slice(None, -1))
ramp = x_now.sub(x_prev, join="override")
ramp

LinearExpression [time: 4]:
---------------------------
[1]: +1 x[1] - 1 x[0]
[2]: +1 x[2] - 1 x[1]
[3]: +1 x[3] - 1 x[2]
[4]: +1 x[4] - 1 x[3]

In [ ]:
# Alternative: filter absent slots with .sel() after shift
shifted = (1 * x).shift(time=1)
valid = ~shifted.isnull()
shifted.sel(time=valid)

LinearExpression [time: 4]:
---------------------------
[1]: +1 x[0]
[2]: +1 x[1]
[3]: +1 x[2]
[4]: +1 x[3]

---

## Reviving absent slots

Addition/subtraction automatically fill const with 0 (additive identity) — this is not arbitrary, it preserves associativity.

For multiplication/division, NaN propagates. To revive absent slots before multiplying:

- **`.fillna(value)`** — fill before arithmetic. Works on both Variables and Expressions. `Variable.fillna(numeric)` returns a `LinearExpression`.
- **`.mul(value, fill_value=)`** — fill and multiply in one step.

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()
shifted = (1 * x).shift(time=1)

# Multiplication propagates NaN — absent stays absent
shifted * 3

In [ ]:
# fillna(0) revives, then multiplication works
shifted.fillna(0) * 3

In [ ]:
# Shorthand: .mul(value, fill_value=) does both in one step
shifted.mul(3, fill_value=0)

In [ ]:
# Variable.fillna(numeric) returns a LinearExpression
x.shift(time=1).fillna(0)

### Outer join with fill_value

When combining expressions with mismatched coordinates, absent terms on each side don't poison valid terms. The outer join preserves all coordinates.

In [ ]:
m = Model()
tech_a, tech_b = ["wind", "solar"], ["solar", "gas"]
cap_a = m.add_variables(lower=0, coords=[tech_a], name="cap_a")
cap_b = m.add_variables(lower=0, coords=[tech_b], name="cap_b")
cost_a = xr.DataArray([10, 20], coords=[("dim_0", tech_a)])
cost_b = xr.DataArray([15, 25], coords=[("dim_0", tech_b)])

# Outer join: each tech keeps its valid terms, absent terms are ignored at solve time
combined = (cap_a * cost_a).add(cap_b * cost_b, join="outer")
print("isnull:", combined.isnull().values)
combined

---

## FILL_VALUE internals

| Type | Field | FILL_VALUE | Why |
|---|---|---|---|
| LinearExpression | `vars` | -1 | Integer sentinel (no variable) |
| LinearExpression | `coeffs` | NaN | Absent — not a numeric value |
| LinearExpression | `const` | NaN | Absent — needed for `isnull()` detection |
| Variable | `labels` | -1 | Integer sentinel (no variable) |
| Variable | `lower` | NaN | Absent bound |
| Variable | `upper` | NaN | Absent bound |

All float fields use NaN for absence. Integer fields use -1.

In [ ]:
print("FILL_VALUE:", FILL_VALUE)

FILL_VALUE: {'vars': -1, 'coeffs': nan, 'const': nan}


In [ ]:
linopy.options.reset()